# 000 — Jesse data basics

This notebook inspects candle data; it does **not** define a trading strategy. Run `make setup`, select the **Quant Research Lab** kernel, and place a small CSV under `data/raw/` or use the optional Jesse database cell below. No market data is committed to Git.

## Candle conventions

OHLCV means open, high, low, close, and volume. Jesse arrays use `[timestamp, open, close, high, low, volume]`; our pandas table uses the more familiar `open, high, low, close, volume` order and a UTC timestamp index.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analytics import clean_ohlcv, forward_returns, log_returns, rolling_volatility, simple_returns

## Load and audit a CSV

Expected columns: `timestamp, open, high, low, close, volume`. Timestamps may be ISO-8601 strings or Unix milliseconds. Inspect duplicates and missing values before cleaning so quality problems remain visible.

In [ ]:
DATA_PATH = Path('../../data/raw/candles.csv')

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Add a candle CSV at {DATA_PATH.resolve()} or change DATA_PATH.')

raw = pd.read_csv(DATA_PATH)
raw.head()

In [ ]:
print('Missing values by column:')
display(raw.isna().sum())
print('Duplicated raw timestamps:', raw['timestamp'].duplicated().sum())

candles = clean_ohlcv(raw)
print('Rows after cleaning:', len(candles))
print('Sorted:', candles.index.is_monotonic_increasing)
print('Duplicate UTC timestamps:', candles.index.duplicated().sum())
candles.head()

## Optional: load candles already imported into Jesse

Start Jesse and import a supported exchange-symbol first. `get_candles` returns `(warmup, trading)` arrays. Keep this cell commented until the database and requested dates exist. Jesse aggregates its stored 1-minute candles into the requested timeframe.

In [ ]:
# import os
# os.environ['POSTGRES_HOST'] = 'localhost'  # notebook runs on the WSL host
# os.environ['POSTGRES_PORT'] = '5434'
# os.environ['REDIS_HOST'] = 'localhost'
# os.environ['REDIS_PORT'] = '6380'
# import jesse.helpers as jh
# from jesse.research import get_candles
# _, values = get_candles(
#     'Binance Spot', 'BTC-USDT', '1h',
#     jh.date_to_timestamp('2024-01-01'),
#     jh.date_to_timestamp('2024-02-01'),
# )
# raw = pd.DataFrame(values, columns=['timestamp', 'open', 'close', 'high', 'low', 'volume'])
# candles = clean_ohlcv(raw)

## Returns and rolling volatility

Simple returns measure percentage change. Log returns add across time. Forward returns are labels for later observations and must never become same-timestamp strategy inputs. The annualization factor below assumes 24/7 hourly crypto data; change it for the actual market calendar.

In [ ]:
analysis = candles.copy()
analysis['simple_return'] = simple_returns(analysis['close'])
analysis['log_return'] = log_returns(analysis['close'])
for horizon in range(1, 5):
    analysis[f'forward_return_{horizon}'] = forward_returns(analysis['close'], horizon)
analysis['rolling_volatility_24'] = rolling_volatility(analysis['log_return'], window=24)
analysis.tail()

In [ ]:
analysis[['close', 'simple_return', 'log_return', 'rolling_volatility_24']].describe().T

## Visual checks

Plots are diagnostic: look for gaps, jumps, regime changes, bad timestamps, and volatility clusters before interpreting summary statistics.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
analysis['close'].plot(ax=axes[0], title='Close price')
analysis['simple_return'].plot(ax=axes[1], title='Hourly simple returns', linewidth=0.8)
axes[0].set_ylabel('Price')
axes[1].set_ylabel('Return')
plt.tight_layout()

## Questions to answer manually

- What timezone and session does this dataset represent?
- Are missing timestamps expected market closures or data gaps?
- Were duplicates identical, or did cleaning discard conflicting records?
- Are prices adjusted for splits and dividends where applicable?
- Which volatility annualization factor matches this market?